# 5. AIND Ephys Curation

Run [aind-ephys-curation](https://github.com/AllenNeuralDynamics/aind-ephys-curation/tree/main/code) on the postprocessed analyzer from `04_aind_ephys_postprocessing.ipynb`.

The capsule does two things per recording:
1. Default QC pass/fail using the query in `params.json` (default: `isi_violations_ratio < 0.5 and presence_ratio > 0.8 and amplitude_cutoff < 0.1`). Saved to `qc_<name>.npy`.
2. Auto-labelling each unit as `noise` / `sua` / `mua` via two HuggingFace models (`SpikeInterface/UnitRefine_noise_neural_classifier`, `SpikeInterface/UnitRefine_sua_mua_classifier`). Saved to `unit_classifier_<name>.csv`.

It expects `../data/` to contain `postprocessed_<name>.zarr` directories and writes outputs to `../results/`.

## Imports and deps

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "huggingface-hub", "skops",
        # The SpikeInterface/UnitRefine HuggingFace models were trained with
        # sklearn 1.5.x. Pin to that range so the pickled SimpleImputer loads.
        "scikit-learn==1.5.2",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv


Resolved 41 packages in 243ms


 Downloaded scikit-learn
Prepared 1 package in 9.92s
Uninstalled 3 packages in 107ms
Installed 3 packages in 9ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2
 - scikit-learn==1.8.0
 + scikit-learn==1.5.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'huggingface-hub', 'skops', 'scikit-learn==1.5.2'], returncode=0)

## Clone the capsule and seed `data/`

In [3]:
cur_repo = Path("/tmp/aind-ephys-curation")
if not cur_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-ephys-curation.git",
            str(cur_repo),
        ],
        check=True,
    )

data_dir = cur_repo / "data"
results_dir = cur_repo / "results"
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

for stale in list(data_dir.iterdir()) + list(results_dir.iterdir()):
    shutil.rmtree(stale) if stale.is_dir() else stale.unlink()

post_results = Path.cwd() / "postprocessing_results"
assert post_results.exists(), "Run 04_aind_ephys_postprocessing.ipynb first."

for entry in post_results.iterdir():
    if entry.name.startswith("postprocessed_") and entry.suffix == ".zarr":
        dest = data_dir / entry.name
        shutil.copytree(entry, dest)

print("Seeded data dir:")
for p in sorted(data_dir.iterdir()):
    print(" ", p.name)

Cloning into '/tmp/aind-ephys-curation'...


Seeded data dir:
  postprocessed_block0_None_recording1.zarr


## Use the upstream params unchanged

The default query operates on `isi_violations_ratio`, `presence_ratio`, and `amplitude_cutoff` — all of which the postprocessing zarr already contains.

In [4]:
upstream_params = json.loads((cur_repo / "code" / "params.json").read_text())
params = {**upstream_params, "job_kwargs": {**upstream_params.get("job_kwargs", {}), "n_jobs": 1}}
params_path = cur_repo / "code" / "params_toy.json"
params_path.write_text(json.dumps(params, indent=2))
print("Query:", params["query"])
print("Wrote", params_path)

Query: isi_violations_ratio < 0.5 and presence_ratio > 0.8 and amplitude_cutoff < 0.1
Wrote /tmp/aind-ephys-curation/code/params_toy.json


## Run the curation capsule

First time only: this downloads two small skops classifiers from HuggingFace.

In [5]:
argv = [
    "python", "-u", "run_capsule.py",
    "--params", params_path.name,
    "--n-jobs", "1",
]
print("Running:", " ".join(argv))
result = subprocess.run(
    argv,
    cwd=cur_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"curation run failed with code {result.returncode}")

Running: python -u run_capsule.py --params params_toy.json --n-jobs 1



CURATION
Curating recording: block0_None_recording1
Curation query: isi_violations_ratio < 0.5 and presence_ratio > 0.8 and amplitude_cutoff < 0.1
	Passing default QC: 0 / 10
Applying noise-neural classifier from SpikeInterface/UnitRefine_noise_neural_classifier
HTTP Request: GET https://huggingface.co/api/models/SpikeInterface/UnitRefine_noise_neural_classifier/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/SpikeInterface/UnitRefine_noise_neural_classifier/resolve/main/best_model.skops "HTTP/1.1 302 Found"
HTTP Request: HEAD https://huggingface.co/SpikeInterface/UnitRefine_noise_neural_classifier/resolve/main/metadata.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/SpikeInterface/UnitRefine_noise_neural_classifier/280cd07ff8310286ded5c0cfdb382e92e2a43a98/metadata.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/SpikeInterface/UnitRefine_noise_neural_classifier/

## Copy outputs next to the notebook

In [6]:
local_results_dir = Path.cwd() / "curation_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
shutil.copytree(results_dir, local_results_dir)

for p in sorted(local_results_dir.iterdir()):
    rel = p.relative_to(local_results_dir)
    kind = "dir" if p.is_dir() else f"{p.stat().st_size} bytes"
    print(f"  {rel}  ({kind})")

  data_process_curation_block0_None_recording1.json  (1429 bytes)
  qc_block0_None_recording1.npy  (138 bytes)
  unit_classifier_block0_None_recording1.csv  (261 bytes)


## Inspect QC flags and unit classifications

In [7]:
import numpy as np
import pandas as pd

qc_files = sorted(local_results_dir.glob("qc_*.npy"))
csv_files = sorted(local_results_dir.glob("unit_classifier_*.csv"))

for qc_path in qc_files:
    flags = np.load(qc_path)
    print(f"{qc_path.name}: {int(flags.sum())} / {len(flags)} units pass default QC")

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    print()
    print(csv_path.name)
    print(df)

qc_block0_None_recording1.npy: 0 / 10 units pass default QC

unit_classifier_block0_None_recording1.csv
  decoder_label  decoder_probability
0           sua             0.513499
1           sua             0.639853
2           mua             0.621799
3           mua             0.518425
4           mua             0.619274
5           sua             0.505997
6           mua             0.548739
7           mua             0.502879
8           sua             0.643502
9           mua             0.572040
